# Mejoras para ai-chat-guardrails


Este proyecto puede caer como mejora de arquitectura o tests, incluso sin IA real.

Hallazgo real al probar:

- `uv run pytest` no encuentra `chatbot` si no se configura el path.
- con `PYTHONPATH=.` los tests corren, pero fallan por mensajes esperados en inglés frente a mensajes reales en español.


In [ ]:
# Mejora 1: añadir esto al pyproject.toml del proyecto ai-chat-guardrails.
# Así pytest encuentra el paquete sin tener que usar PYTHONPATH=.

"""
[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]
"""


In [ ]:
# Mejora 2: hacer asserts menos frágiles si el idioma puede cambiar.
# En vez de comprobar frases exactas en inglés, comprueba comportamiento.

"""
def test_check_length():
    ok, _ = input_guard.validate("Hello world", llm_judge=dummy_judge_safe)
    assert ok

    ok, reason = input_guard.validate("   ", llm_judge=dummy_judge_safe)
    assert not ok
    assert reason

    long_msg = "a" * 501
    ok, reason = input_guard.validate(long_msg, llm_judge=dummy_judge_safe)
    assert not ok
    assert "501" in reason
"""


In [ ]:
# Mejora 3: arreglar base_url en factory.py.
# La config define base_url, pero el factory no se lo pasa a OllamaBackend.

"""
return OllamaBackend(
    model=config.model_name,
    system_prompt=system_prompt,
    base_url=config.base_url,
)
"""


In [ ]:
# Mejora 4: persistir historial al cerrar.
# Útil si piden una mejora funcional sencilla.

"""
import json
from pathlib import Path

class ChatEngine:
    ...
    def save_history(self, path: str = "history.json") -> None:
        Path(path).write_text(json.dumps(self.history, ensure_ascii=False, indent=2), encoding="utf-8")

    def load_history(self, path: str = "history.json") -> None:
        history_path = Path(path)
        if history_path.exists():
            self.history = json.loads(history_path.read_text(encoding="utf-8"))
            self._trim_history()
"""


Explicación corta:

“Esta mejora hace el proyecto más robusto en entorno de examen: los tests se ejecutan sin variables manuales, las comprobaciones no dependen del idioma exacto del mensaje, Ollama respeta la URL configurada y el historial puede persistirse.”
